import pandas as pd
import numpy as np

In [2]:
import pandas as pd
import numpy as np

In [3]:
pharm_data = pd.read_csv("C:\\Users\\USER\\Downloads\\bloom_nigeria_pharma_cleaned_v1.csv")
pharm_data.head()

,Facility_ID,Facility_Name,State,LGA,Geopolitical_Zone,Facility_Type,Medicine_Name,Medicine_Category,WHO_Essential_List,Batch_Number,...,Serialization_Code,Supply_Chain_Visibility,Stockout_Events_Last_6Months,Average_Monthly_Consumption,Wastage_Percentage_Last_Quarter,Quality_Verified,Regulatory_Compliance,Distribution_Lead_Time_Days,Warehouse_Temp_Log_Available,Status
0,FAC-JIG-5012,General Hospital Central,Jigawa,Central,North_West,General Hospital,Ibuprofen 400mg,Analgesic,Yes,BN21395,...,SRL-F7078BDA71,Full,0,594,5.9,Yes,WHO_PQ,31,Yes,Expired
1,FAC-OND-1711,West PHC,Ondo,West,South_West,PHC,Nevirapine 200mg,ARV,Yes,BN92397,...,NaN,NaN,2,258,7.1,Yes,WHO_PQ,8,No,In_Stock
2,FAC-OYO-1916,General Hospital Rural B,Oyo,Rural B,South_West,General Hospital,Nevirapine 200mg,ARV,Yes,BN84341,...,NaN,NaN,2,291,8.1,Yes,WHO_PQ,9,Yes,Expired
3,FAC-AKW-1188,Ward 11 PHC,Akwa Ibom,West,South_South,PHC,Measles-Rubella Vaccine,Vaccine,Yes,BN94012,...,NaN,NaN,2,539,41.4,Pending,NAFDAC,5,No,In_Stock
4,FAC-FCT-2403,Federal Teaching Hospital Abaji,Fct,Abaji,North_Central,Teaching Hospital,Morphine 10mg/Ml Injection,Analgesic,Yes,BN26828,...,NaN,NaN,3,726,7.1,Yes,WHO_PQ,42,No,Expired


In [4]:
pharm_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 37 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Facility_ID                      2000 non-null   str    
 1   Facility_Name                    2000 non-null   str    
 2   State                            2000 non-null   str    
 3   LGA                              2000 non-null   str    
 4   Geopolitical_Zone                2000 non-null   str    
 5   Facility_Type                    2000 non-null   str    
 6   Medicine_Name                    2000 non-null   str    
 7   Medicine_Category                2000 non-null   str    
 8   WHO_Essential_List               2000 non-null   str    
 9   Batch_Number                     2000 non-null   str    
 10  Manufacturing_Date               2000 non-null   str    
 11  Expiry_Date                      2000 non-null   str    
 12  Manufacturer                   

In [5]:
pharm_data.describe()

,Current_Stock_Level,Reorder_Level,Max_Capacity,Days_Of_Stock_Remaining,Unit_Cost_Naira,Total_Stock_Value,Stockout_Events_Last_6Months,Average_Monthly_Consumption,Wastage_Percentage_Last_Quarter,Distribution_Lead_Time_Days
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2.000000e+03,2000.000000,2000.000000,2000.000000,2000.000000
mean,1024.703000,616.632000,3308.373000,75.454650,1855.223825,2.240964e+06,2.967500,411.251000,10.007150,19.559000
std,1381.395144,333.313166,2171.726588,82.043464,3224.993995,7.755928e+06,2.263627,222.210339,11.065775,12.701267
min,-49.000000,30.000000,80.000000,0.000000,20.520000,0.000000e+00,0.000000,20.000000,1.000000,3.000000
25%,57.000000,328.000000,1546.000000,7.600000,211.630000,1.984007e+04,1.000000,219.000000,4.200000,9.000000
50%,451.000000,624.000000,2956.000000,41.400000,833.600000,2.317522e+05,3.000000,416.000000,7.100000,15.000000
75%,1434.750000,897.250000,4716.000000,121.225000,2261.715000,1.376013e+06,4.000000,598.250000,10.500000,30.000000
max,8890.000000,1200.000000,9588.000000,351.800000,21971.490000,1.264895e+08,8.000000,800.000000,83.600000,45.000000


In [ ]:
# Fix negative stock levels
pharm_data['Current_Stock_Level'] = pharm_data['Current_Stock_Level'].clip(lower=0)

# Verify
print(pharm_data['Current_Stock_Level'].min())

0


In [7]:
# Parse all date columns
date_cols = ['Manufacturing_Date', 'Expiry_Date', 'Last_Restocked_Date', 'Next_Expected_Delivery']

for col in date_cols:
    pharm_data[col] = pd.to_datetime(pharm_data[col], dayfirst=True, errors='coerce')

# Check
pharm_data[date_cols].dtypes

Manufacturing_Date        datetime64[us]
Expiry_Date               datetime64[us]
Last_Restocked_Date       datetime64[us]
Next_Expected_Delivery    datetime64[us]
dtype: object

In [8]:
# Fix Status where stock is 0 but not marked Out_of_Stock
mask = (pharm_data['Current_Stock_Level'] == 0) & (pharm_data['Status'] != 'Out_of_Stock')
print(f"Records to fix: {mask.sum()}")

pharm_data.loc[mask, 'Status'] = 'Out_of_Stock'

Records to fix: 219


In [9]:
# Fix Status where Expiry_Date has passed but Status != Expired
today = pd.Timestamp('2026-03-10')
mask = (pharm_data['Expiry_Date'] < today) & (pharm_data['Status'] != 'Expired')
print(f"Records to fix: {mask.sum()}")

pharm_data.loc[mask, 'Status'] = 'Expired'

Records to fix: 365


In [12]:
# Flag duplicate Facility IDs
pharm_data['Data_Quality_Flag'] = pharm_data.duplicated(subset='Facility_ID', keep=False)
print(pharm_data['Data_Quality_Flag'].value_counts())

Data_Quality_Flag
False    1777
True      223
Name: count, dtype: int64


In [14]:
# Missing/nulls
print(pharm_data.isnull().sum())

# Zeros in stock
print(f"Zero stock records: {(pharm_data['Current_Stock_Level'] == 0).sum()}")

Facility_ID                           0
Facility_Name                         0
State                                 0
LGA                                   0
Geopolitical_Zone                     0
Facility_Type                         0
Medicine_Name                         0
Medicine_Category                     0
WHO_Essential_List                    0
Batch_Number                          0
Manufacturing_Date                 1305
Expiry_Date                        1286
Manufacturer                          0
Current_Stock_Level                   0
Reorder_Level                         0
Max_Capacity                          0
Unit                                  0
Days_Of_Stock_Remaining               0
Last_Restocked_Date                1300
Next_Expected_Delivery                0
Supplier_Name                       410
Unit_Cost_Naira                       0
Total_Stock_Value                     0
Storage_Condition_Required            0
Actual_Storage_Condition              0


In [16]:
# Check original values that failed to parse
raw = pd.read_csv("C:\\Users\\USER\\Downloads\\bloom_nigeria_pharma_cleaned_v1.csv")
print(raw['Expiry_Date'][pharm_data['Expiry_Date'].isnull()].value_counts().head(20))

Expiry_Date
2025-01-23    7
2023-11-24    6
2024-08-25    6
2025-02-27    5
2025-04-27    5
2026-06-16    5
2025-10-16    5
2024-05-28    5
2025-12-26    5
2025-06-19    5
2025-01-29    5
2025-12-18    4
2024-11-19    4
2025-06-22    4
2024-12-29    4
2026-04-16    4
2024-12-28    4
2026-08-18    4
2026-01-31    4
2026-12-17    4
Name: count, dtype: int64


In [18]:
date_cols = ['Manufacturing_Date', 'Expiry_Date', 'Last_Restocked_Date', 'Next_Expected_Delivery']

for col in date_cols:
    pharm_data[col] = pd.to_datetime(
        pd.read_csv("C:\\Users\\USER\\Downloads\\bloom_nigeria_pharma_cleaned_v1.csv")[col], 
        dayfirst=True, 
        errors='coerce'
    )

print(pharm_data[date_cols].isnull().sum())

Manufacturing_Date        1305
Expiry_Date               1286
Last_Restocked_Date       1300
Next_Expected_Delivery       0
dtype: int64


In [19]:
raw = pd.read_csv("C:\\Users\\USER\\Downloads\\bloom_nigeria_pharma_cleaned_v1.csv")
print(raw['Manufacturing_Date'].isnull().sum())
print(raw['Expiry_Date'].isnull().sum())
print(raw['Last_Restocked_Date'].isnull().sum())

0
0
0


In [20]:
from dateutil import parser

def parse_date(val):
    try:
        return parser.parse(str(val), dayfirst=True)
    except:
        return pd.NaT

date_cols = ['Manufacturing_Date', 'Expiry_Date', 'Last_Restocked_Date', 'Next_Expected_Delivery']

for col in date_cols:
    pharm_data[col] = raw[col].apply(parse_date)

print(pharm_data[date_cols].isnull().sum())

Manufacturing_Date        0
Expiry_Date               0
Last_Restocked_Date       0
Next_Expected_Delivery    0
dtype: int64


In [21]:
mask = (pharm_data['Expiry_Date'] > today) & (pharm_data['Status'] == 'Expired')
print(f"Future expiry but marked Expired: {mask.sum()}")
print(pharm_data[mask][['Facility_ID', 'Medicine_Name', 'Expiry_Date', 'Status']].head(10))

Future expiry but marked Expired: 0
Empty DataFrame
Columns: [Facility_ID, Medicine_Name, Expiry_Date, Status]
Index: []


In [22]:
# Calculate expected days of stock
pharm_data['Expected_Days_Stock'] = (
    pharm_data['Current_Stock_Level'] / pharm_data['Average_Monthly_Consumption'] * 30
).round(1)

# Compare with actual
pharm_data['Days_Stock_Discrepancy'] = (
    pharm_data['Days_Of_Stock_Remaining'] - pharm_data['Expected_Days_Stock']
).abs()

print(f"Records with >10 day discrepancy: {(pharm_data['Days_Stock_Discrepancy'] > 10).sum()}")
print(pharm_data[pharm_data['Days_Stock_Discrepancy'] > 10][
    ['Medicine_Name', 'Current_Stock_Level', 'Average_Monthly_Consumption', 
     'Days_Of_Stock_Remaining', 'Expected_Days_Stock', 'Days_Stock_Discrepancy']
].head(10))

Records with >10 day discrepancy: 0
Empty DataFrame
Columns: [Medicine_Name, Current_Stock_Level, Average_Monthly_Consumption, Days_Of_Stock_Remaining, Expected_Days_Stock, Days_Stock_Discrepancy]
Index: []


In [23]:
# Calculate expected total value
pharm_data['Expected_Stock_Value'] = (
    pharm_data['Current_Stock_Level'] * pharm_data['Unit_Cost_Naira']
).round(2)

# Compare with actual
pharm_data['Value_Discrepancy'] = (
    pharm_data['Total_Stock_Value'] - pharm_data['Expected_Stock_Value']
).abs()

print(f"Records with >100 naira discrepancy: {(pharm_data['Value_Discrepancy'] > 100).sum()}")
print(pharm_data[pharm_data['Value_Discrepancy'] > 100][
    ['Medicine_Name', 'Current_Stock_Level', 'Unit_Cost_Naira',
     'Total_Stock_Value', 'Expected_Stock_Value', 'Value_Discrepancy']
].head(10))

Records with >100 naira discrepancy: 0
Empty DataFrame
Columns: [Medicine_Name, Current_Stock_Level, Unit_Cost_Naira, Total_Stock_Value, Expected_Stock_Value, Value_Discrepancy]
Index: []


In [25]:
#checking for inconsistencies in medicine names
print(pharm_data['Medicine_Name'].nunique())
print(sorted(pharm_data['Medicine_Name'].unique()))


29
['Amoxicillin 250mg Caps', 'Amoxicillin 250mg Capsules', 'Amoxicillin-Clavulanate 625mg', 'Artemether-Lumefantrine 20/120mg', 'Artesunate Inj. 60mg', 'Artesunate Injection 60mg', 'Azithromycin 500mg', 'Bcg Vaccine', 'Ceftriaxone 1G Inj.', 'Ceftriaxone 1G Injection', 'Chloroquine Phosphate 250mg', 'Ciprofloxacin 500mg', 'Diclofenac 50mg', 'Dihydroartemisinin-Piperaquine', 'Efavirenz/Lamivudine/Tenofovir', 'Hpv Vaccine', 'Ibuprofen 400mg', 'Lopinavir/Ritonavir 200/50mg', 'Measles-Rubella Vaccine', 'Metronidazole 400mg', 'Morphine 10mg/Ml Inj.', 'Morphine 10mg/Ml Injection', 'Nevirapine 200mg', 'Opv (Oral Polio Vaccine)', 'Paracetamol 500mg', 'Pentavalent Vaccine (Dpt-Hepb-Hib)', 'Tenofovir/Lamivudine/Dolutegravir', 'Tramadol 50mg', 'Yellow Fever Vaccine']


In [26]:
#standardising the medicine names for consistent formatting
name_mapping = {
    'Amoxicillin 250mg Caps': 'Amoxicillin 250mg Capsules',
    'Artesunate Inj. 60mg': 'Artesunate Injection 60mg',
    'Ceftriaxone 1G Inj.': 'Ceftriaxone 1G Injection',
    'Morphine 10mg/Ml Inj.': 'Morphine 10mg/Ml Injection'
}

pharm_data['Medicine_Name'] = pharm_data['Medicine_Name'].replace(name_mapping)

print(pharm_data['Medicine_Name'].nunique()) 

25


In [27]:
mask = (pharm_data['Days_Of_Stock_Remaining'] > 0) & (pharm_data['Status'] == 'Out_of_Stock')
print(f"Records with stock days but marked Out_of_Stock: {mask.sum()}")
print(pharm_data[mask][['Medicine_Name', 'Current_Stock_Level', 'Days_Of_Stock_Remaining', 'Status']].head(10))

Records with stock days but marked Out_of_Stock: 0
Empty DataFrame
Columns: [Medicine_Name, Current_Stock_Level, Days_Of_Stock_Remaining, Status]
Index: []


In [28]:
mask = pharm_data['Reorder_Level'] > pharm_data['Max_Capacity']
print(f"Reorder level exceeds max capacity: {mask.sum()}")
print(pharm_data[mask][['Medicine_Name', 'Reorder_Level', 'Max_Capacity']].head(10))

Reorder level exceeds max capacity: 0
Empty DataFrame
Columns: [Medicine_Name, Reorder_Level, Max_Capacity]
Index: []


In [29]:
mask = pharm_data['Manufacturing_Date'] > pharm_data['Expiry_Date']
print(f"Manufacturing date after expiry date: {mask.sum()}")
print(pharm_data[mask][['Medicine_Name', 'Manufacturing_Date', 'Expiry_Date']].head(10))

Manufacturing date after expiry date: 0
Empty DataFrame
Columns: [Medicine_Name, Manufacturing_Date, Expiry_Date]
Index: []


In [30]:
# Check 1 - Expiring_Within_30Days status accuracy
mask1 = pharm_data['Status'] == 'Expiring_Within_30Days'
actually_expiring = (
    (pharm_data['Expiry_Date'] > today) & 
    (pharm_data['Expiry_Date'] <= today + pd.Timedelta(days=30))
)
print("=== CHECK 1: Expiring_Within_30Days accuracy ===")
print(f"Marked as Expiring_Within_30Days: {mask1.sum()}")
print(f"Actually expiring within 30 days: {actually_expiring.sum()}")
print(f"Mismatch: {(mask1 & ~actually_expiring).sum()}")

# Check 2 - Low_Stock vs Reorder_Level
print("\n=== CHECK 2: Low_Stock vs Reorder_Level ===")
mask2 = pharm_data['Status'] == 'Low_Stock'
actually_low = pharm_data['Current_Stock_Level'] < pharm_data['Reorder_Level']
print(f"Marked as Low_Stock: {mask2.sum()}")
print(f"Actually below reorder level: {actually_low.sum()}")
print(f"Mismatch: {(mask2 & ~actually_low).sum()}")

# Check 3 - Critical status threshold
print("\n=== CHECK 3: Critical status ===")
mask3 = pharm_data['Status'] == 'Critical'
print(f"Marked as Critical: {mask3.sum()}")
print(pharm_data[mask3][['Current_Stock_Level', 'Reorder_Level', 'Days_Of_Stock_Remaining']].describe())

# Check 4 - Wastage outliers
print("\n=== CHECK 4: Wastage outliers ===")
print(f"Wastage > 50%: {(pharm_data['Wastage_Percentage_Last_Quarter'] > 50).sum()}")
print(pharm_data[pharm_data['Wastage_Percentage_Last_Quarter'] > 50][
    ['Medicine_Name', 'Facility_Name', 'State', 'Wastage_Percentage_Last_Quarter']
].head(10))

# Check 5 - Manufacturing_Date > Last_Restocked_Date
print("\n=== CHECK 5: Manufacturing after restock ===")
mask5 = pharm_data['Manufacturing_Date'] > pharm_data['Last_Restocked_Date']
print(f"Manufacturing date after last restock: {mask5.sum()}")

# Check 6 - Last_Restocked_Date > Next_Expected_Delivery
print("\n=== CHECK 6: Last restock after next delivery ===")
mask6 = pharm_data['Last_Restocked_Date'] > pharm_data['Next_Expected_Delivery']
print(f"Last restock date after next expected delivery: {mask6.sum()}")

=== CHECK 1: Expiring_Within_30Days accuracy ===
Marked as Expiring_Within_30Days: 35
Actually expiring within 30 days: 59
Mismatch: 35

=== CHECK 2: Low_Stock vs Reorder_Level ===
Marked as Low_Stock: 127
Actually below reorder level: 1056
Mismatch: 0

=== CHECK 3: Critical status ===
Marked as Critical: 117
       Current_Stock_Level  Reorder_Level  Days_Of_Stock_Remaining
count           117.000000     117.000000               117.000000
mean            166.008547     654.076923                11.523932
std             134.623645     329.923219                 6.632972
min               2.000000      79.000000                 0.100000
25%              54.000000     364.000000                 5.700000
50%             123.000000     676.000000                11.600000
75%             268.000000     912.000000                17.100000
max             494.000000    1186.000000                22.400000

=== CHECK 4: Wastage outliers ===
Wastage > 50%: 45
                          Medicin

In [31]:
#fix
mask = (
    (pharm_data['Expiry_Date'] > today) & 
    (pharm_data['Expiry_Date'] <= today + pd.Timedelta(days=30)) &
    (pharm_data['Status'] != 'Expiring_Within_30Days')
)
print(f"Records to update: {mask.sum()}")
pharm_data.loc[mask, 'Status'] = 'Expiring_Within_30Days'

Records to update: 59


In [32]:
print(pharm_data[mask][['Medicine_Name', 'Expiry_Date', 'Status']].head(20))

                          Medicine_Name Expiry_Date                  Status
11       Dihydroartemisinin-Piperaquine  2026-03-16  Expiring_Within_30Days
13         Lopinavir/Ritonavir 200/50mg  2026-03-12  Expiring_Within_30Days
38                     Nevirapine 200mg  2026-03-19  Expiring_Within_30Days
39                      Ibuprofen 400mg  2026-04-03  Expiring_Within_30Days
81       Efavirenz/Lamivudine/Tenofovir  2026-03-17  Expiring_Within_30Days
106       Amoxicillin-Clavulanate 625mg  2026-03-20  Expiring_Within_30Days
166                 Metronidazole 400mg  2026-03-16  Expiring_Within_30Days
253                 Ciprofloxacin 500mg  2026-03-14  Expiring_Within_30Days
309                 Metronidazole 400mg  2026-03-16  Expiring_Within_30Days
322                 Metronidazole 400mg  2026-03-18  Expiring_Within_30Days
438                     Ibuprofen 400mg  2026-03-25  Expiring_Within_30Days
457         Chloroquine Phosphate 250mg  2026-03-15  Expiring_Within_30Days
462  Pentava

In [33]:
mask = pharm_data['Manufacturing_Date'] > pharm_data['Last_Restocked_Date']
print(pharm_data[mask][['Medicine_Name', 'Manufacturer', 'Manufacturing_Date', 'Last_Restocked_Date']].head(10))

                       Medicine_Name           Manufacturer  \
11    Dihydroartemisinin-Piperaquine      Fidson Healthcare   
47                 Paracetamol 500mg      Fidson Healthcare   
318      Chloroquine Phosphate 250mg      Greenfield Pharma   
620                Paracetamol 500mg    May & Baker Nigeria   
866                      Hpv Vaccine      Greenfield Pharma   
893                Paracetamol 500mg          Cipla Nigeria   
1067      Amoxicillin 250mg Capsules  Emzor Pharmaceuticals   
1137      Morphine 10mg/Ml Injection     Jawa International   
1138      Morphine 10mg/Ml Injection    May & Baker Nigeria   
1205   Amoxicillin-Clavulanate 625mg          Roche Nigeria   

     Manufacturing_Date Last_Restocked_Date  
11           2024-04-16          2024-03-12  
47           2024-12-28          2024-10-12  
318          2024-12-20          2024-08-12  
620          2024-12-20          2024-08-12  
866          2024-09-12          2024-04-12  
893          2024-12-06       

In [34]:
mask = pharm_data['Manufacturing_Date'] > pharm_data['Last_Restocked_Date']
pharm_data.loc[mask, 'Last_Restocked_Date'] = pd.NaT
print(f"Nulled out: {mask.sum()} impossible restock dates")

Nulled out: 18 impossible restock dates


In [35]:
mask = pharm_data['Last_Restocked_Date'] > pharm_data['Next_Expected_Delivery']
print(pharm_data[mask][['Medicine_Name', 'Last_Restocked_Date', 'Next_Expected_Delivery']].head(10))

                         Medicine_Name Last_Restocked_Date  \
1                     Nevirapine 200mg          2025-07-03   
8             Ceftriaxone 1G Injection          2025-08-02   
12         Chloroquine Phosphate 250mg          2025-09-01   
20                       Tramadol 50mg          2025-09-02   
23                 Metronidazole 400mg          2025-09-02   
33        Lopinavir/Ritonavir 200/50mg          2025-08-01   
38                    Nevirapine 200mg          2025-07-01   
44           Artesunate Injection 60mg          2025-11-03   
49  Pentavalent Vaccine (Dpt-Hepb-Hib)          2025-08-02   
50      Efavirenz/Lamivudine/Tenofovir          2025-08-05   

   Next_Expected_Delivery  
1              2025-06-15  
8              2025-07-18  
12             2025-07-13  
20             2025-06-26  
23             2025-06-24  
33             2025-07-05  
38             2025-06-08  
44             2025-06-14  
49             2025-06-26  
50             2025-07-10  


In [36]:
mask = pharm_data['Last_Restocked_Date'] > pharm_data['Next_Expected_Delivery']
pharm_data.loc[mask, 'Next_Expected_Delivery'] = pd.NaT
print(f"Nulled out: {mask.sum()} outdated delivery dates")

Nulled out: 320 outdated delivery dates


In [37]:
print(pharm_data[pharm_data['Wastage_Percentage_Last_Quarter'] > 50][
    ['Medicine_Name', 'Facility_Name', 'State', 'Wastage_Percentage_Last_Quarter', 'Storage_Condition_Required', 'Cold_Chain_Compliant']
].sort_values('Wastage_Percentage_Last_Quarter', ascending=False).head(10))

                           Medicine_Name                      Facility_Name  \
99              Opv (Oral Polio Vaccine)        South Primary Health Centre   
805                          Hpv Vaccine                           East PHC   
698                          Hpv Vaccine                   Kaduna South PHC   
163                          Hpv Vaccine                        Central PHC   
1684  Pentavalent Vaccine (Dpt-Hepb-Hib)                         Ward 9 PHC   
52               Measles-Rubella Vaccine      Rural A Primary Health Centre   
305                          Hpv Vaccine  Enugu South Primary Health Centre   
1673             Measles-Rubella Vaccine      Bungudu Primary Health Centre   
140                          Bcg Vaccine     Potiskum Primary Health Centre   
534              Measles-Rubella Vaccine              General Hospital Kaga   

        State  Wastage_Percentage_Last_Quarter Storage_Condition_Required  \
99      Delta                             83.6       

In [38]:
pharm_data.drop(columns=['Expected_Days_Stock', 'Days_Stock_Discrepancy', 
                          'Expected_Stock_Value', 'Value_Discrepancy'], inplace=True)

pharm_data.to_csv("C:\\Users\\USER\\Downloads\\bloom_nigeria_pharma_cleaned_v2.csv", index=False)
print("Saved!")

Saved!
